# Kalliope SRU Abfrage und parsen der Briefwechsel von Werner Heisenberg
Sources: Code f+r die SRU-Abfrage und das Parsen der Daten adaptiert nach DNB SRU Tutorial:

https://github.com/deutsche-nationalbibliothek/dnblab/blob/main/DNB_SRU_Tutorial.ipynb 

Dokumentation Kalliope SRU:
https://kalliope-verbund.info/de/support/sru.html

Zweck ist es, die Daten so auszulesen, dass im Anschluß für eine Netzwerkanalyse weiterverarbeitet werden können

In [1]:
# Import necessary libraries
# Benötigte Bibliotheken importieren

import requests
from lxml import etree
import pandas as pd

In [4]:
# Function to send a SRU request to the Kalliope system with a custom query
# Funktion zum Senden einer SRU-Anfrage an das Kalliope-System mit einer benutzerdefinierten Abfrage
# SRU query
def kalliope_sru(query):
    base_url = "https://kalliope-verbund.info/sru"
    params = {
        'version': '1.2',
        'operation': 'searchRetrieve',
        'recordSchema': 'mods37',
        'maximumRecords': '1500',   #mehr records werden gefunden, wenn maximum records hochgesetzt wird.
        'query': query
    }
    
    r = requests.get(base_url, params=params)
    mods_content = r.content
    records_mods = etree.fromstring(mods_content)
    
    return records_mods


In [5]:
# Function to parse a MODS record (SRU response XML) and extract relevant fields
# Funktion zum Parsen eines MODS-Datensatzes (SRU XML-Antwort) und Extraktion relevanter Felder
def parse_mods(record):
    ns = {
        'srw': 'http://www.loc.gov/zing/srw/',  # SRW namespace
        'mods': 'http://www.loc.gov/mods/v3'    # MODS namespace
    }
    
    # Extract RecordID
    recordIdentifier = record.xpath(".//mods:mods/mods:recordInfo/mods:recordIdentifier", namespaces=ns)
    recordIdentifier = recordIdentifier[0].text if recordIdentifier else "unknown"
    
    # Extract Title
    title = record.xpath(".//mods:mods/mods:titleInfo/mods:title", namespaces=ns)
    title = title[0].text if title else "unknown"
    
    # Extract Date
    date = record.xpath(".//mods:mods/mods:originInfo/mods:dateCreated", namespaces=ns)
    date = date[0].text if date else "unknown"
    
    # Extract Names and Roles
    names = record.xpath(".//mods:mods/mods:name", namespaces=ns)
    senders = []
    receivers = []
    mentioned = []
    
    for name in names:
        name_text = name.xpath(".//mods:namePart/text()", namespaces=ns)
        role_text = name.xpath(".//mods:role/mods:roleTerm[@type='text']/text()", namespaces=ns)
        
        if name_text and role_text:
            name_value = name_text[0]
            role_value = role_text[0].lower()
            
            # Categorize based on role
            if "verfasser" in role_value:  # Adjust to match actual role values
                senders.append(name_value)
            elif "adressat" in role_value:
                receivers.append(name_value)
            elif "erwähnt" in role_value:
                mentioned.append(name_value)
    
    # Extract Genre
    genre_letter = record.xpath(".//mods:mods/mods:genre[text()='Brief']", namespaces=ns)
    genre_letter = genre_letter[0].text if genre_letter else "unknown"
    
    # Return a dictionary to build the DataFrame
    return {
        "recordIdentifier": recordIdentifier,
        "title": title,
        "date": date,
        "senders": " /".join(senders),    # Combine names into a single string
        "receivers": "/ ".join(receivers),
        "mentioned": "/ ".join(mentioned),
        "genre": genre_letter
    }


In [6]:
# Define the SRU query string, e.g. to retrieve letters from a specific archive or person
# Definieren der SRU-Abfragezeichenkette, z. B. um Briefe aus einem bestimmten Archiv oder von bestimmten Personen abzurufen
# Example query

#query = 'ead.archdesc.id="DE-611-BF-73161"'

#query = 'ead.archdesc.id="DE-611-BF-73161" and ead.unitdate_end<1980'

query = 'ead.archdesc.id="DE-611-BF-73161" AND ead.unitdate_start>=1945 AND ead.unitdate_end<=1950'

#'ead.archdesc.id'
#query = "ead.addressee"=="Heisenberg"
records_xml = kalliope_sru(query)

print(f'{len(records_xml.xpath("//srw:record", namespaces={"srw": "http://www.loc.gov/zing/srw/"}))} Ergebnisse gefunden')


1500 Ergebnisse gefunden


In [7]:
# Parse the retrieved XML records and convert them into a list of dictionaries
# Parsen der abgerufenen XML-Datensätze und Umwandlung in eine Liste von Dictionaries
# Parse data and convert to DataFrame
records = records_xml.xpath("//srw:record", namespaces={"srw": "http://www.loc.gov/zing/srw/"})
output = [parse_mods(record) for record in records]
df = pd.DataFrame(output)
df


,recordIdentifier,title,date,senders,receivers,mentioned,genre
0,DE-611-HS-4020260,"Brief von H. K. Müller an Werner Heisenberg, 2...",1947-02-27,"Müller, H. K. (1910-)","Heisenberg, Werner (1901-1976)",,Brief
1,DE-611-HS-4020266,"Brief von Werner Heisenberg an Rudolf Müller, ...",1946-06-26,"Heisenberg, Werner (1901-1976)","Müller, Rudolf",,Brief
2,DE-611-HS-4020269,"Brief von Rudolf Müller an Werner Heisenberg, ...",1946-06-10,"Müller, Rudolf","Heisenberg, Werner (1901-1976)",,Brief
3,DE-611-HS-4020276,Brief von Werner Heisenberg an Kurt Müller-Lüb...,1947-09-29,"Heisenberg, Werner (1901-1976)","Müller-Lübeck, Kurt",,Brief
4,DE-611-HS-4020283,Brief von Werner Heisenberg an Kurt Müller-Lüb...,1947-03-07,"Heisenberg, Werner (1901-1976)","Müller-Lübeck, Kurt","Grabbe, Ingrid",Brief
...,...,...,...,...,...,...,...
1495,DE-611-HS-3731376,Brief von Hans Seeliger von Max-Planck-Gesells...,1949-02,"Seeliger, Hans (1908-) /Max-Planck-Gesellschaf...","Heisenberg, Werner (1901-1976)",,Brief
1496,DE-611-HS-3732724,Brief von Werner Heisenberg an Otto Hahn an Ma...,1949-03-03,"Heisenberg, Werner (1901-1976)","Hahn, Otto (1879-1968)/ Max-Planck-Gesellschaf...","Weizsäcker, Carl Friedrich von (1912-2007)/ Wi...",Brief
1497,DE-611-HS-3732727,Brief von Kurt Pfuhl von Max-Planck-Gesellscha...,1949-04-25,"Pfuhl, Kurt /Max-Planck-Gesellschaft zur Förde...","Heisenberg, Werner (1901-1976)/ Max-Planck-Ins...","Dieminger, Walter (1907-2000)",Brief
1498,DE-611-HS-3732729,Brief von Werner Heisenberg an Ernst Telschow ...,1949-04-26,"Heisenberg, Werner (1901-1976)","Telschow, Ernst (1889-1988)/ Max-Planck-Gesell...","Dieminger, Walter (1907-2000)/ Pfuhl, Kurt",Brief


In [1]:
# Optional: print raw XML for inspection
# Optional: Rohes XML zur Überprüfung ausgeben
#print(etree.tostring(records_xml, pretty_print=True).decode())

In [8]:
# Save full parsed data as CSV file
# Gespeicherte, vollständig geparste Daten als CSV-Datei
df.to_csv("heisenberg_1945-1950.csv", index=False) # adjust filename according to query

In [9]:
#df_bibsonomy_Europa_publications = df_bibsonomy_Europa.loc[df_bibsonomy_Europa['type'] == 'Publication']
df_B = df.loc[df["genre"] == "Brief"]
df_B

,recordIdentifier,title,date,senders,receivers,mentioned,genre
0,DE-611-HS-4020260,"Brief von H. K. Müller an Werner Heisenberg, 2...",1947-02-27,"Müller, H. K. (1910-)","Heisenberg, Werner (1901-1976)",,Brief
1,DE-611-HS-4020266,"Brief von Werner Heisenberg an Rudolf Müller, ...",1946-06-26,"Heisenberg, Werner (1901-1976)","Müller, Rudolf",,Brief
2,DE-611-HS-4020269,"Brief von Rudolf Müller an Werner Heisenberg, ...",1946-06-10,"Müller, Rudolf","Heisenberg, Werner (1901-1976)",,Brief
3,DE-611-HS-4020276,Brief von Werner Heisenberg an Kurt Müller-Lüb...,1947-09-29,"Heisenberg, Werner (1901-1976)","Müller-Lübeck, Kurt",,Brief
4,DE-611-HS-4020283,Brief von Werner Heisenberg an Kurt Müller-Lüb...,1947-03-07,"Heisenberg, Werner (1901-1976)","Müller-Lübeck, Kurt","Grabbe, Ingrid",Brief
...,...,...,...,...,...,...,...
1495,DE-611-HS-3731376,Brief von Hans Seeliger von Max-Planck-Gesells...,1949-02,"Seeliger, Hans (1908-) /Max-Planck-Gesellschaf...","Heisenberg, Werner (1901-1976)",,Brief
1496,DE-611-HS-3732724,Brief von Werner Heisenberg an Otto Hahn an Ma...,1949-03-03,"Heisenberg, Werner (1901-1976)","Hahn, Otto (1879-1968)/ Max-Planck-Gesellschaf...","Weizsäcker, Carl Friedrich von (1912-2007)/ Wi...",Brief
1497,DE-611-HS-3732727,Brief von Kurt Pfuhl von Max-Planck-Gesellscha...,1949-04-25,"Pfuhl, Kurt /Max-Planck-Gesellschaft zur Förde...","Heisenberg, Werner (1901-1976)/ Max-Planck-Ins...","Dieminger, Walter (1907-2000)",Brief
1498,DE-611-HS-3732729,Brief von Werner Heisenberg an Ernst Telschow ...,1949-04-26,"Heisenberg, Werner (1901-1976)","Telschow, Ernst (1889-1988)/ Max-Planck-Gesell...","Dieminger, Walter (1907-2000)/ Pfuhl, Kurt",Brief


In [10]:
# Optional: save only letter-related records
# Optional: nur briefbezogene Datensätze speichern
df_B.to_csv("heisenberg_1945-1950_lettersonly.csv", index=False) # adjust filename according to query

In [11]:
# ToDo: further clean and filter the data (e.g., by sender/recipient, date)
# Noch zu tun: Daten weiter bereinigen und filtern (z. B. nach Absender/Empfänger, Datum)
# next steps to clean data: pick columns date, senders, receivers from df and think
# about how to deal with separators in order to cleary separate the columns
#df_bibsonomy_Europa_selection = df_bibsonomy_Europa_publications[
#   ["type", "id", "tags", "label", "user", "description", "date", "authors", "publisher", "isbn"]] 


df_l_selec = df_B[["date","senders", "receivers"]]
df_l_selec

,date,senders,receivers
0,1947-02-27,"Müller, H. K. (1910-)","Heisenberg, Werner (1901-1976)"
1,1946-06-26,"Heisenberg, Werner (1901-1976)","Müller, Rudolf"
2,1946-06-10,"Müller, Rudolf","Heisenberg, Werner (1901-1976)"
3,1947-09-29,"Heisenberg, Werner (1901-1976)","Müller-Lübeck, Kurt"
4,1947-03-07,"Heisenberg, Werner (1901-1976)","Müller-Lübeck, Kurt"
...,...,...,...
1495,1949-02,"Seeliger, Hans (1908-) /Max-Planck-Gesellschaf...","Heisenberg, Werner (1901-1976)"
1496,1949-03-03,"Heisenberg, Werner (1901-1976)","Hahn, Otto (1879-1968)/ Max-Planck-Gesellschaf..."
1497,1949-04-25,"Pfuhl, Kurt /Max-Planck-Gesellschaft zur Förde...","Heisenberg, Werner (1901-1976)/ Max-Planck-Ins..."
1498,1949-04-26,"Heisenberg, Werner (1901-1976)","Telschow, Ernst (1889-1988)/ Max-Planck-Gesell..."


In [14]:
# Save final selection with names and dates to a CSV file
# Speichere finale Auswahl mit Namen und Daten in eine CSV-Datei

df_l_selec.to_csv("heisenberg_namesdates_1945-1950.csv", index=False, sep=";") # adjust filename according to query